# Step 07.2 — Embedding Geometry：长度、距离、方向与相似度

7.1 已经知道每个 token 会查到一个 `C` 维向量。

现在的问题变成：**两个 embedding vector 到底怎样比较？**

这一节完整解决 5 个问题：

1. 向量的长度（L2 norm）和方向分别是什么意思；
2. Euclidean distance 在比较什么；
3. Dot product 为什么既看方向、又看长度；
4. Cosine similarity 为什么可以只比较方向；
5. 怎样构造 pairwise similarity matrix，并正确解释 embedding 可视化。

## 1. 一个向量同时有“长度”和“方向”

先只看二维，方便画图。

例如：

```text
a = [3,4]
```

把它看成从原点 `(0,0)` 指向 `(3,4)` 的箭头。

L2 norm 定义：

$$
\|a\|_2 = \sqrt{a_1^2+a_2^2+\cdots+a_C^2}
$$

对于 `[3,4]`，长度是 5。

In [ ]:
import mlx.core as mx

a = mx.array([3.0, 4.0])
b = mx.array([6.0, 8.0])

def l2_norm(x):
    return mx.sqrt(mx.sum(x * x))

print("a      =", a)
print("||a||2 =", l2_norm(a).item())
print("b      =", b)
print("||b||2 =", l2_norm(b).item())

`b=[6,8]` 正好是 `a` 的 2 倍，所以它们方向完全相同，但长度分别是 5 和 10。

### 为什么单位化以后只留下方向？

$$
\hat{x}=\frac{x}{\|x\|_2}
$$

这是把每个分量都除以同一个正数，所以分量比例不变，向量不会转向，只会被缩放到长度 1。

例如：

```text
[3,4] -> [0.6,0.8]
[6,8] -> [0.6,0.8]
```

零向量是例外：它的 norm 为 0，没有定义方向。

In [ ]:
a_hat = a / l2_norm(a)
b_hat = b / l2_norm(b)

mx.eval(a_hat, b_hat)
print("a_hat =", a_hat, "norm =", l2_norm(a_hat).item())
print("b_hat =", b_hat, "norm =", l2_norm(b_hat).item())
print("same direction after normalization:", bool(mx.allclose(a_hat, b_hat)))

### 把长度和方向画出来

`ax.arrow()` 可以从原点画向量；`ax.set_aspect('equal')` 让两个坐标轴比例一致，避免视觉上扭曲角度。

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
for vec, label in [([3, 4], "a=[3,4]"), ([6, 8], "b=[6,8]"), ([0.6, 0.8], "unit direction")]:
    ax.arrow(0, 0, vec[0], vec[1], length_includes_head=True, head_width=0.18)
    ax.annotate(label, (vec[0], vec[1]), xytext=(5, 5), textcoords="offset points")

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_xlim(-1, 7)
ax.set_ylim(-1, 9)
ax.set_aspect("equal")
ax.set_xlabel("dimension 0")
ax.set_ylabel("dimension 1")
ax.set_title("Same direction, different magnitudes")
plt.show()

## 2. Euclidean Distance：两个点在空间里离多远？

如果把 embedding 看成空间中的点：

$$
d(a,b)=\|a-b\|_2
$$

就是先求位移向量 `a-b`，再求这个位移的长度。

Euclidean distance 同时受到方向差异和 magnitude 差异影响。

In [ ]:
def euclidean_distance(x, y):
    return l2_norm(x - y)

p = mx.array([1.0, 1.0])
q = mx.array([2.0, 1.0])
r = mx.array([5.0, 5.0])

print("distance(p,q) =", euclidean_distance(p, q).item())
print("distance(p,r) =", euclidean_distance(p, r).item())
print("distance(a,b) =", euclidean_distance(a, b).item())

特别注意：`a=[3,4]` 与 `b=[6,8]` 明明同方向，但 Euclidean distance 仍是 5。

所以“方向相似”和“空间位置接近”不是同一个概念。

## 3. Dot Product：既看夹角，也会被向量长度放大

$$
a\cdot b=\sum_i a_i b_i
$$

更重要的几何公式：

$$
\boxed{a\cdot b=\|a\|\|b\|\cos\theta}
$$

所以 dot product 同时依赖两个 norm 和两向量夹角。

In [ ]:
def dot(x, y):
    return mx.sum(x * y)

u = mx.array([1.0, 0.0])
v = mx.array([10.0, 0.0])
w = mx.array([1.0, 1.0])

print("u·v =", dot(u, v).item())
print("u·w =", dot(u, w).item())

`u` 与 `v` 完全同方向，但因为 `v` 很长，dot product 是 10。

所以：

> **dot product 不能直接叫 cosine similarity。**

以后 Self-Attention 的 `QK^T` 用的就是 dot product。除非 Q/K 预先归一化，否则 score 同时受方向和 norm 影响。

## 4. Cosine Similarity：除掉长度，只比较方向

$$
\boxed{\cos\theta=\frac{a\cdot b}{\|a\|\|b\|}}
$$

对非零向量：

```text
+1 -> 完全同方向
 0 -> 正交 / 90°
-1 -> 完全反方向
```

零向量没有定义方向，所以 cosine similarity 也未定义。

In [ ]:
def cosine_similarity(x, y):
    nx = l2_norm(x)
    ny = l2_norm(y)
    if nx.item() == 0 or ny.item() == 0:
        raise ValueError("cosine similarity is undefined for a zero vector")
    return dot(x, y) / (nx * ny)

base = mx.array([1.0, 0.0])
same = mx.array([100.0, 0.0])
perpendicular = mx.array([0.0, 2.0])
opposite = mx.array([-3.0, 0.0])

print("same direction:", cosine_similarity(base, same).item())
print("perpendicular :", cosine_similarity(base, perpendicular).item())
print("opposite      :", cosine_similarity(base, opposite).item())

若先归一化：

```text
a_hat = a / ||a||
b_hat = b / ||b||
```

则 `||a_hat||=||b_hat||=1`，所以：

$$
a_{hat}\cdot b_{hat}=\cos\theta
$$

这就是“单位化后 dot product 只比较方向”的严格原因。

In [ ]:
x = mx.array([3.0, 4.0])
y = mx.array([4.0, 3.0])

x_hat = x / l2_norm(x)
y_hat = y / l2_norm(y)

cos_xy = cosine_similarity(x, y)
normalized_dot = dot(x_hat, y_hat)

mx.eval(cos_xy, normalized_dot)
print("cosine(x,y)      =", cos_xy.item())
print("dot(unit,unit)   =", normalized_dot.item())
print("same            =", bool(mx.allclose(cos_xy, normalized_dot)))

因此 Euclidean distance、dot product、cosine similarity 没有一个永远最好；它们回答的是不同的几何问题。

## 5. Pairwise Similarity Matrix：一次观察整个 embedding table

下面为了教学手工造 5 个三维向量，让 `cat/dog`、`car/truck` 比较接近。**这不是训练结果，只是示意。**

In [ ]:
tokens = ["cat", "dog", "sun", "car", "truck"]

E = mx.array([
    [ 1.0,  0.9,  0.1],
    [ 0.9,  1.0,  0.2],
    [-0.2,  0.1,  1.0],
    [-0.9, -0.8,  0.1],
    [-1.0, -0.7,  0.0],
])

norms = mx.sqrt(mx.sum(E * E, axis=1, keepdims=True))
E_hat = E / norms
cosine_matrix = E_hat @ E_hat.T
mx.eval(cosine_matrix)

print("E shape:", E.shape)
print("cosine matrix shape:", cosine_matrix.shape)
print(cosine_matrix)

Shape 推导：

```text
E_hat       : (V,C)
E_hat.T     : (C,V)
(V,C)@(C,V): (V,V)
```

矩阵 `(i,j)` 就是 token i 与 token j 的 cosine similarity。

这也是以后理解 `QK^T` 生成 pairwise score matrix 的前置练习。

In [ ]:
matrix_for_plot = cosine_matrix.tolist()

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(matrix_for_plot, vmin=-1, vmax=1)
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_title("Toy pairwise cosine similarity matrix")

for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f"{matrix_for_plot[i][j]:.2f}", ha="center", va="center")

fig.colorbar(image, ax=ax, label="cosine similarity")
plt.show()

### 怎样解释这张图，怎样不能解释？

可以说：在这组向量和这个 metric 下，某些 token 的方向更相似。

不能直接说：模型一定“理解了”某个人类概念。

真实 learned embedding 也一样：它由训练目标塑造，相似结构可能出现，但不保证符合人类概念分类；不同层的 hidden state 与原始 token embedding 也不是同一个空间；PCA/t-SNE/UMAP 等降维图只能作为分析线索。

尤其我们的字符级 MiniGPT vocabulary 很小，不要期待字符 embedding 自动形成漂亮、可命名的语义 cluster。

### 7.2 最终结论

- L2 norm `||x||` 表示向量长度；`x/||x||` 将非零向量缩放到单位长度而保持方向；
- Euclidean distance `||a-b||` 比较绝对空间距离，因此受方向和 magnitude 共同影响；
- dot product `a·b = ||a||||b||cosθ` 同时依赖 norm 与夹角；
- cosine similarity 去掉 norm，只比较方向；零向量的 cosine 未定义；
- 只有向量归一化后，dot product 才等于 cosine similarity；
- `E_hat @ E_hat.T` 可以一次得到 `(V,V)` pairwise cosine matrix；
- embedding geometry 是训练目标产生的内部结构，不应把可视化邻近关系过度解释成人类语义证明；
- 下一节 7.3：从整个训练 batch 出发，把 `(B,T)` token IDs 完整转换成 `(B,T,C)`，并明确 `C` 为什么会成为后续整个 Transformer 的 model width。